In [ ]:
from sklearn.model_selection import train_test_split

X_train_before_tfidf, X_test_before_tfidf, y_train, y_test = train_test_split(
    atlanta_rest["processed_text"], 
    atlanta_rest["categoryName"], 
    test_size=0.2, 
    random_state=42
)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 1. Initialize TF-IDF Vectorizer with Uni, Bi, and Tri-grams
tfidf = TfidfVectorizer(
    ngram_range=(1, 3), 
    min_df=5,             
    max_df=0.8,           
    max_features=25000,  
    sublinear_tf=True    
)

# 2. Fit and Transform the Training Data

X_train = tfidf.fit_transform(X_train_before_tfidf)

# 3. Transform the Test Data
X_test = tfidf.transform(X_test_before_tfidf)

# 4. Inspect the resulting shapes and features
print("Train TF-IDF shape:", X_train.shape)
print("Test TF-IDF shape:", X_test.shape)

feature_names = np.array(tfidf.get_feature_names_out())

# Print some random examples of features to check if trigrams are present
print("\nExample features (including trigrams):")
# We grab a slice from the middle/end where longer n-grams might reside
print(feature_names[15000:15010])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize Logistic Regression with 'balanced' class weights
# This automatically assigns higher weights to minority classes (e.g., 'New American')
# and lower weights to majority classes (e.g., 'Mexican')
clf = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    class_weight='balanced',  
    random_state=42          
)

# Train the model
clf.fit(X_train, y_train)

# Predict on the test set
y_pred = clf.predict(X_test)

# Evaluate model performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

# Generate confusion matrix
# We use clf.classes_ to ensure the labels match the order learned by the model
labels_sorted = clf.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
print("\nConfusion matrix shape:", cm.shape)

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize SMOTE
# random_state ensures reproducibility.
smote = SMOTE(random_state=42)

# 2. Apply SMOTE to the training data ONLY
# This generates synthetic samples for minority classes to balance the dataset.
# IMPORTANT: We never apply SMOTE to the test set to avoid data leakage.
print(f"Original training shape: {X_train.shape}")
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f"Resampled training shape: {X_train_resampled.shape}")

# 3. Initialize the Classifier
# Note: We do NOT need class_weight='balanced' anymore because SMOTE 
# has already balanced the classes physically.
clf_smote = LogisticRegression(
    max_iter=2000, 
    n_jobs=-1, 
    random_state=42
)

# 4. Train the model on the resampled (balanced) data
clf_smote.fit(X_train_resampled, y_train_resampled)

# 5. Predict on the original test set
# We use the original test data to evaluate real-world performance.
y_pred_smote = clf_smote.predict(X_test)

# 6. Evaluate
print("Accuracy (with SMOTE):", accuracy_score(y_test, y_pred_smote))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred_smote))

# Optional: Confusion Matrix to see where the model is confused
labels_sorted = clf_smote.classes_
cm = confusion_matrix(y_test, y_pred_smote, labels=labels_sorted)
print(f"\nConfusion Matrix Shape: {cm.shape}")

In [ ]:
# Random Forest on TF-IDF features (note: TF-IDF is sparse; we densify a reduced feature set)

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Reduce dimensionality before Random Forest (keeps it feasible)
k = 3000  # tune: 1000-10000 depending on your data size
selector = SelectKBest(score_func=chi2, k=k)

X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

# Convert to dense arrays (RandomForest typically expects dense input)
X_train_dense = X_train_sel.toarray()
X_test_dense = X_test_sel.toarray()

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_dense, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test_dense)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Initialize Linear Support Vector Classifier (SVM)
# SVMs usually work better than Logistic Regression for high-dimensional text data (TF-IDF)
# class_weight='balanced' is still useful here
clf_svm = LinearSVC(
    class_weight='balanced', 
    random_state=42,
    dual=False,     
    max_iter=3000
)

# Train
clf_svm.fit(X_train, y_train)

# Predict
y_pred_svm = clf_svm.predict(X_test)

# Evaluate
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nSVM Classification report:\n")
print(classification_report(y_test, y_pred_svm))

In [ ]:
# === 1. Define the Custom Taxonomy ===
# Dictionary mapping specific restaurant types to your 7 defined meta-classes
meta_mapping = {
    'Mexican restaurant': 'Mexican',
    
    'Italian restaurant': 'European',
    'Pizza restaurant': 'European',
    'Mediterranean restaurant': 'European',
    
    'Chinese restaurant': 'Asian',
    'Japanese restaurant': 'Asian',
    'Thai restaurant': 'Asian',
    'Indian restaurant': 'Asian',
    
    'American restaurant': 'American',
    'New American restaurant': 'American',
    'Bar & grill': 'American',
    'Hamburger restaurant': 'American',
    
    'Fast food restaurant': 'Fast_Food',
    'Sandwich shop': 'Fast_Food',
    'Chicken restaurant': 'Fast_Food',
    'Breakfast restaurant': 'Fast_Food',
    
    'Steak house': 'Meat_BBQ',
    'Barbecue restaurant': 'Meat_BBQ',
    
    'Seafood restaurant': 'Seafood'
}

# === 2. Apply Mapping ===
# Create a new column 'meta_category' using the dictionary
atlanta_rest['meta_category'] = atlanta_rest['categoryName'].map(meta_mapping)

# === 3. Validation ===
# Check if any rows were missed (should be 0)
missing_count = atlanta_rest['meta_category'].isna().sum()
print(f"Rows without a meta-category: {missing_count}")

# Print the distribution of the new meta-classes to check balance
print("\nDistribution of new Meta-Categories:")
print(atlanta_rest['meta_category'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# We need both the specific label (y_fine) and the broad group (y_meta)
X = atlanta_rest["processed_text"]
y_fine = atlanta_rest["categoryName"]    # 19 classes
y_meta = atlanta_rest["meta_category"]   # 7 classes

# 1. Split the data
# We stratify by the fine-grained category to ensure all small classes are represented
X_train_raw, X_test_raw, y_fine_train, y_fine_test, y_meta_train, y_meta_test = train_test_split(
    X, y_fine, y_meta, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_fine
)

# 2. Vectorization (TF-IDF)
# Using Uni-Bi-Trigrams as discussed
tfidf = TfidfVectorizer(
    ngram_range=(1, 3), 
    min_df=5, 
    max_df=0.8, 
    max_features=25000, 
    sublinear_tf=True
)

X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

print(f"Train shape: {X_train.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import numpy as np

# Dictionary to store our sub-classifiers
sub_classifiers = {}

# === STEP A: TRAIN THE PARENT MODEL (META-CLASSIFIER) ===
# This model learns to distinguish between "Asian", "European", "Mexican", etc.
print("Training Parent Model (Meta-Classifier)...")

clf_parent = LinearSVC(
    class_weight='balanced', 
    random_state=42,
    dual=False 
)
clf_parent.fit(X_train, y_meta_train)


# === STEP B: TRAIN CHILD MODELS (SUB-CLASSIFIERS) ===
unique_meta_groups = y_meta_train.unique()

for group in unique_meta_groups:
    
    # 1. Find indices of samples belonging to this group in the training set
    # FIX: We add .values or .to_numpy() to convert the Pandas Series to a Numpy array.
    # Scipy sparse matrices require Numpy arrays for boolean indexing.
    indices = (y_meta_train == group).values
    
    # 2. Get the specific sub-labels for this group
    # Pandas Series (y_fine_train) can handle numpy array indexing fine.
    sub_labels = y_fine_train[indices]
    
    # 3. Check how many unique sub-classes are in this group
    unique_sub_classes = sub_labels.unique()
    
    # If the group has only 1 class, we don't need a sub-model.
    if len(unique_sub_classes) <= 1:
        print(f"Skipping sub-model for '{group}' (only 1 subclass: {unique_sub_classes[0]})")
        sub_classifiers[group] = "is_leaf" 
        continue
        
    # 4. Train a specific classifier for this group
    print(f"Training sub-model for group '{group}' with classes: {unique_sub_classes}")
    
    sub_clf = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        n_jobs=-1,
        random_state=42
    )
    
    # Now X_train[indices] works because indices is a numpy array
    sub_clf.fit(X_train[indices], sub_labels)
    
    # Store the trained model in our dictionary
    sub_classifiers[group] = sub_clf

print("Training complete.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def hierarchical_predict(parent_model, sub_models_dict, X_data):
    """
    Custom function to predict classes hierarchically.
    """
    # 1. Predict the Meta-Category for all samples
    meta_predictions = parent_model.predict(X_data)
    
    final_predictions = []
    
    # 2. Iterate through each prediction and refine it
    # Note: iterating row-by-row is slow for massive data, but fine for 10k rows.
    # For production, we would use masking (vectorized approach).
    for i, meta_pred in enumerate(meta_predictions):
        
        # Get the row vector
        row_vector = X_data[i]
        
        # Check if we have a sub-model for this group
        if meta_pred in sub_models_dict:
            handler = sub_models_dict[meta_pred]
            
            if handler == "is_leaf":
                # If it's a leaf group (e.g., Mexican), the meta-prediction implies the specific class
                # We need to map the group name back to the specific class name manually or assume logic
                # For this specific dataset logic:
                if meta_pred == 'Mexican':
                    final_predictions.append('Mexican restaurant')
                elif meta_pred == 'Seafood':
                    final_predictions.append('Seafood restaurant')
                else:
                    # Fallback
                    final_predictions.append(meta_pred) 
            else:
                # Use the sub-model to predict the fine-grained class
                sub_pred = handler.predict(row_vector)[0]
                final_predictions.append(sub_pred)
        else:
            # Fallback if something goes wrong (shouldn't happen)
            final_predictions.append(meta_pred)
            
    return np.array(final_predictions)

# === RUN PREDICTION ===
y_pred_hierarchical = hierarchical_predict(clf_parent, sub_classifiers, X_test)

# === EVALUATE ===
print("Hierarchical Accuracy:", accuracy_score(y_fine_test, y_pred_hierarchical))
print("\nClassification Report:\n")
print(classification_report(y_fine_test, y_pred_hierarchical))